# تحليل بيانات مبيعات متجر إلكتروني
## E-commerce Sales Data Analysis

---

**الهدف من المشروع:**  
تحليل بيانات مبيعات متجر إلكتروني تحتوي على أكثر من 2500 عملية بيع، بهدف:
- تنظيف البيانات وجعلها جاهزة للتحليل
- اكتشاف الأنماط والاتجاهات في المبيعات
- تحديد الفئات والمنتجات الأعلى ربحية
- تقديم رؤى قابلة للتنفيذ لدعم قرارات العمل

**الأدوات المستخدمة:**
- Python 3
- Pandas, NumPy (معالجة البيانات)
- Matplotlib, Seaborn (التصور البصري)
- Jupyter Notebook

---

## 1. استيراد المكتبات وتحميل البيانات

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configure visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Custom color palette
colors = ['#2E4057', '#048A81', '#54C6EB', '#8A89C0', '#CDA7C7']
sns.set_palette(colors)

print('Libraries imported successfully')

In [ ]:
# Load the data
df = pd.read_csv('sales_data_raw.csv')

print(f'Dataset shape: {df.shape}')
print(f'Number of rows: {df.shape[0]:,}')
print(f'Number of columns: {df.shape[1]}')
df.head()

## 2. استكشاف البيانات الأولي (Initial Data Exploration)

In [ ]:
# General information about the dataset
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing,
    'Percentage (%)': missing_percent.round(2)
})
missing_df[missing_df['Missing Values'] > 0]

In [ ]:
# Check for duplicates
print(f'Number of duplicate rows: {df.duplicated().sum()}')

## 3. تنظيف وتجهيز البيانات (Data Cleaning & Preprocessing)

في هذه المرحلة سنقوم بمعالجة المشاكل التي اكتشفناها:
1. القيم المفقودة
2. الصفوف المكررة
3. عدم اتساق صيغ التواريخ
4. عدم اتساق حالة الأحرف في الأعمدة النصية

In [ ]:
# Save a copy of original data for comparison
df_original = df.copy()

# 3.1 Remove duplicates
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'Removed {before - after} duplicate rows')
print(f'Dataset now has {after} rows')

In [ ]:
# 3.2 Handle missing values
# Strategy: Fill categorical missing values with the mode (most frequent value)

categorical_cols = ['Customer_Segment', 'Region', 'Shipping_Mode']

for col in categorical_cols:
    mode_value = df[col].mode()[0]
    df[col] = df[col].fillna(mode_value)
    print(f'Filled {col} missing values with: {mode_value}')

print(f'\nRemaining missing values:\n{df.isnull().sum()}')

In [ ]:
# 3.3 Fix inconsistent date formats
def parse_date(date_str):
    """Convert dates to a unified format"""
    try:
        return pd.to_datetime(date_str, format='%Y-%m-%d')
    except:
        try:
            return pd.to_datetime(date_str, format='%d/%m/%Y')
        except:
            return pd.NaT

df['Order_Date'] = df['Order_Date'].apply(parse_date)
print(f'All dates converted. Sample:')
print(df['Order_Date'].head())

In [ ]:
# 3.4 Standardize text capitalization
text_cols = ['Region', 'Customer_Segment', 'Category', 'Product', 'Shipping_Mode']

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

print('Text columns standardized (Title Case)')
print(f'\nUnique regions: {df["Region"].unique()}')

In [ ]:
# 3.5 Create additional useful columns
df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
df['Month_Name'] = df['Order_Date'].dt.month_name()
df['Day_of_Week'] = df['Order_Date'].dt.day_name()
df['Quarter'] = df['Order_Date'].dt.quarter

# Calculate profit margin
df['Profit_Margin'] = (df['Profit'] / df['Sales'] * 100).round(2)

print('New columns added: Year, Month, Month_Name, Day_of_Week, Quarter, Profit_Margin')
df.head()

In [ ]:
# Data cleaning summary
print('='*50)
print('DATA CLEANING SUMMARY')
print('='*50)
print(f'Original rows: {len(df_original):,}')
print(f'Cleaned rows: {len(df):,}')
print(f'Duplicates removed: {len(df_original) - len(df):,}')
print(f'Missing values handled: {df_original.isnull().sum().sum()}')
print(f'Date formats unified: Yes')
print(f'Text standardized: Yes')
print(f'New features added: 6 columns')

## 4. التحليل الاستكشافي (Exploratory Data Analysis)

الآن البيانات جاهزة، سنبدأ في استكشاف الأنماط والرؤى.

### 4.1 المؤشرات الرئيسية (Key Performance Indicators)

In [ ]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_orders = len(df)
avg_order_value = df['Sales'].mean()
profit_margin = (total_profit / total_sales) * 100
unique_customers = df['Customer_ID'].nunique()

print('='*50)
print('KEY PERFORMANCE INDICATORS')
print('='*50)
print(f'Total Sales:          ${total_sales:,.2f}')
print(f'Total Profit:         ${total_profit:,.2f}')
print(f'Total Orders:         {total_orders:,}')
print(f'Average Order Value:  ${avg_order_value:,.2f}')
print(f'Profit Margin:        {profit_margin:.2f}%')
print(f'Unique Customers:     {unique_customers:,}')

### 4.2 المبيعات حسب الفئة (Sales by Category)

In [ ]:
category_sales = df.groupby('Category').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order_ID': 'count'
}).round(2).sort_values('Sales', ascending=False)
category_sales.columns = ['Total Sales', 'Total Profit', 'Number of Orders']
category_sales

In [ ]:
# Visualization: Sales by Category
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
category_sales['Total Sales'].plot(kind='bar', ax=axes[0], color=colors[0], edgecolor='black')
axes[0].set_title('Total Sales by Category', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Category', fontsize=12)
axes[0].set_ylabel('Total Sales ($)', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, v in enumerate(category_sales['Total Sales']):
    axes[0].text(i, v + 50000, f'${v/1000:.0f}K', ha='center', fontsize=10, fontweight='bold')

# Pie chart
axes[1].pie(category_sales['Total Sales'], labels=category_sales.index, 
            autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Sales Distribution by Category', fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('images/01_sales_by_category.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.3 اتجاه المبيعات الشهري (Monthly Sales Trend)

In [ ]:
monthly_sales = df.groupby(df['Order_Date'].dt.to_period('M'))['Sales'].sum()
monthly_sales.index = monthly_sales.index.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(monthly_sales.index, monthly_sales.values, marker='o', 
        linewidth=2.5, markersize=8, color=colors[1])
ax.fill_between(monthly_sales.index, monthly_sales.values, alpha=0.2, color=colors[1])

ax.set_title('Monthly Sales Trend (2023-2024)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Total Sales ($)', fontsize=12)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('images/02_monthly_trend.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Best month: {monthly_sales.idxmax().strftime("%B %Y")} with ${monthly_sales.max():,.2f}')
print(f'Worst month: {monthly_sales.idxmin().strftime("%B %Y")} with ${monthly_sales.min():,.2f}')

### 4.4 أعلى 10 منتجات مبيعاً (Top 10 Best-Selling Products)

In [ ]:
top_products = df.groupby('Product')['Sales'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top_products.index[::-1], top_products.values[::-1], 
                color=colors[2], edgecolor='black')

ax.set_title('Top 10 Best-Selling Products', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Total Sales ($)', fontsize=12)
ax.set_ylabel('Product', fontsize=12)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(top_products.values[::-1]):
    ax.text(v + 5000, i, f'${v/1000:.1f}K', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('images/03_top_products.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.5 المبيعات حسب المنطقة (Sales by Region)

In [ ]:
region_analysis = df.groupby('Region').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order_ID': 'count'
}).round(2).sort_values('Sales', ascending=False)
region_analysis.columns = ['Total Sales', 'Total Profit', 'Number of Orders']

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(region_analysis))
width = 0.35

ax.bar(x - width/2, region_analysis['Total Sales']/1000, width, 
        label='Sales ($K)', color=colors[0], edgecolor='black')
ax.bar(x + width/2, region_analysis['Total Profit']/1000, width, 
        label='Profit ($K)', color=colors[1], edgecolor='black')

ax.set_title('Sales and Profit by Region', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Region', fontsize=12)
ax.set_ylabel('Amount (Thousand $)', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(region_analysis.index)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('images/04_sales_by_region.png', dpi=100, bbox_inches='tight')
plt.show()

region_analysis

### 4.6 خريطة حرارية: المبيعات حسب الفئة والمنطقة (Heatmap)

In [ ]:
pivot_table = df.pivot_table(values='Sales', index='Category', 
                               columns='Region', aggfunc='sum').round(2)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot_table, annot=True, fmt=',.0f', cmap='YlGnBu', 
            linewidths=1, linecolor='white', cbar_kws={'label': 'Sales ($)'}, ax=ax)

ax.set_title('Sales Heatmap: Category vs Region', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Region', fontsize=12)
ax.set_ylabel('Category', fontsize=12)

plt.tight_layout()
plt.savefig('images/05_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.7 تأثير الخصومات على الأرباح (Discount Impact on Profit)

In [ ]:
discount_analysis = df.groupby('Discount').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order_ID': 'count'
}).round(2)
discount_analysis.columns = ['Total Sales', 'Total Profit', 'Number of Orders']

fig, ax = plt.subplots(figsize=(12, 6))
ax2 = ax.twinx()

ax.bar(discount_analysis.index.astype(str), discount_analysis['Total Sales'],
        color=colors[0], alpha=0.7, edgecolor='black', label='Sales')
ax2.plot(discount_analysis.index.astype(str), discount_analysis['Total Profit'],
          color=colors[3], marker='o', linewidth=2.5, markersize=10, label='Profit')

ax.set_title('Impact of Discount on Sales and Profit', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Discount Rate', fontsize=12)
ax.set_ylabel('Total Sales ($)', fontsize=12, color=colors[0])
ax2.set_ylabel('Total Profit ($)', fontsize=12, color=colors[3])

ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('images/06_discount_impact.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.8 المبيعات حسب شريحة العميل (Customer Segment Analysis)

In [ ]:
segment_analysis = df.groupby('Customer_Segment').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order_ID': 'count',
    'Customer_ID': 'nunique'
}).round(2)
segment_analysis.columns = ['Total Sales', 'Total Profit', 'Orders', 'Unique Customers']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sales by segment
axes[0].pie(segment_analysis['Total Sales'], labels=segment_analysis.index,
             autopct='%1.1f%%', colors=colors, startangle=90,
             wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Sales by Customer Segment', fontsize=14, fontweight='bold', pad=15)

# Orders by segment
segment_analysis['Orders'].plot(kind='bar', ax=axes[1], color=colors[2], edgecolor='black')
axes[1].set_title('Number of Orders by Segment', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Customer Segment', fontsize=12)
axes[1].set_ylabel('Number of Orders', fontsize=12)
axes[1].tick_params(axis='x', rotation=0)
axes[1].grid(axis='y', alpha=0.3)

for i, v in enumerate(segment_analysis['Orders']):
    axes[1].text(i, v + 20, f'{v}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('images/07_segment_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

segment_analysis

## 5. الخلاصة والتوصيات (Conclusions & Recommendations)

### النتائج الرئيسية:

1. **الفئات الأعلى أداءً:** فئة الإلكترونيات تمثل الحصة الأكبر من المبيعات، مما يستدعي تعزيز المخزون والتسويق لها.

2. **الاتجاهات الموسمية:** توجد ذروات واضحة في المبيعات خلال شهور معينة، يجب استغلالها في التخطيط التسويقي.

3. **أداء المناطق:** بعض المناطق تحقق مبيعات أعلى بشكل ملحوظ، مما يشير إلى فرصة للتوسع في المناطق الأقل أداءً.

4. **تأثير الخصومات:** الخصومات المرتفعة (25%) تقلل من هامش الربح بشكل كبير، يُنصح بضبط استراتيجية الخصومات.

5. **شرائح العملاء:** شريحة Consumer هي الأكثر إنفاقاً، لكن شريحة Corporate تحقق متوسط طلب أعلى.

### التوصيات:

- **التركيز على المنتجات الأعلى ربحية** في الحملات التسويقية
- **إعادة النظر في سياسة الخصومات** لتحسين هامش الربح
- **تطوير استراتيجية تسويقية مخصصة** لكل منطقة بناءً على أدائها
- **الاستثمار في تجربة العملاء** للشرائح الأعلى قيمة
- **الاستعداد للذروات الموسمية** بتخطيط مسبق للمخزون والموظفين

In [ ]:
# Save the cleaned dataset
df.to_csv('sales_data_cleaned.csv', index=False)
print('Cleaned dataset saved as: sales_data_cleaned.csv')
print(f'Final dataset shape: {df.shape}')